**1. Environment Setup**   
First, we'll import the necessary libraries. We use joblib for serialization because it is more efficient than Python's built-in pickle for large NumPy arrays.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

**2. Data Preprocessing & Pipeline Construction**   
In production, data is often "messy." Our pipeline will handle missing values, scale numerical features, and encode categorical features in one flow.

In [ ]:
# Load the dataset (using the common IBM Telco URL)
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Clean TotalCharges (convert string to numeric, handle errors)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)

# Define features and target
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn'].apply(lambda x: 1 if x == 'Yes' else 0)

# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

# Define Preprocessing for Numeric and Categorical data
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

**3. Model Selection & Hyperparameter Tuning**   
We will use a dictionary of models and GridSearchCV to find the best performer. This approach is highly reusable for any classification task.

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the full pipeline with a placeholder for the classifier
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Define the parameter grid
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000)],
        'classifier__C': [0.1, 1, 10]
    },
    {
        'classifier': [RandomForestClassifier()],
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [None, 10, 20]
    }
]

# Run Grid Search
grid_search = GridSearchCV(full_pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")

Best parameters: {'classifier': LogisticRegression(max_iter=1000), 'classifier__C': 10}


**4. Evaluation and Export**   
Once we have the best_estimator_, we evaluate it and export it. This file can then be loaded into a Flask/FastAPI app or a Cloud Function for real-time predictions.

In [ ]:
# Evaluate the best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("--- Model Performance ---")
print(classification_report(y_test, y_pred))

# EXPORT: Save the entire pipeline (preprocessor + model)
joblib.dump(best_model, 'telco_churn_pipeline.pkl')
print("Pipeline exported successfully as 'telco_churn_pipeline.pkl'")

--- Model Performance ---
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.62      0.52      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

Pipeline exported successfully as 'telco_churn_pipeline.pkl'
